Caracterização geral do dataset. Estruturei dados, classes e fontes:

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/content/dataset2_limpo (1).csv")

print("Dimensão:", df.shape)

display(df.head())

print("\nTipos das variáveis:")
display(df.dtypes)

print("\nValores ausentes:")
display(df.isnull().sum())

print("\nDistribuição das classes:")
display(df["classe"].value_counts())

print("\nProporção das classes (%):")
display(df["classe"].value_counts(normalize=True) * 100)

print("\nDistribuição das fontes:")
display(df["fonte"].value_counts())

Dimensão: (3405, 4)


,id,fonte,texto,classe
0,1_0,sms,"Oi, peço para transferir aquele valor para meu...",Legitimate
1,2_0,sms,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate
2,3_0,sms,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate
3,4_0,sms,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate
4,5_0,sms,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate



Tipos das variáveis:


,0
id,object
fonte,object
texto,object
classe,object



Valores ausentes:


...

,count
classe,
Legitimate,2009
Smishing,1396



Proporção das classes (%):


,proportion
classe,
Legitimate,59.001468
Smishing,40.998532



Distribuição das fontes:


,count
fonte,
sms,3005
fb,400


Criar as variáveis que serão analisadas. Visto que as perguntas falam de:

tamanho;
quantidade de palavras;
links;
números;
símbolos;
maiúsculas;
valores monetários;
telefone;
e-mail;
urgência;
dados sensíveis.

Transformei essas características do texto em variáveis:

In [2]:
# Quantidade de caracteres
df["qtd_caracteres"] = df["texto"].str.len()

# Quantidade de palavras
df["qtd_palavras"] = df["texto"].str.split().str.len()

# Quantidade de números/dígitos
df["qtd_numeros"] = df["texto"].apply(
    lambda x: len(re.findall(r"\d", str(x)))
)

# Quantidade de símbolos
df["qtd_simbolos"] = df["texto"].apply(
    lambda x: len(re.findall(r"[^\w\s]", str(x)))
)

# Quantidade de palavras totalmente em maiúsculas
df["qtd_maiusculas"] = df["texto"].apply(
    lambda x: sum(
        1 for palavra in str(x).split()
        if palavra.isupper() and len(palavra) > 1
    )
)

# Possui link
df["possui_link"] = df["texto"].apply(
    lambda x: int(
        bool(re.search(r"(https?://|www\.|bit\.ly|tinyurl)", str(x), re.I))
    )
)

# Possui e-mail
df["possui_email"] = df["texto"].apply(
    lambda x: int(
        bool(re.search(
            r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
            str(x)
        ))
    )
)

# Possui possível telefone
df["possui_telefone"] = df["texto"].apply(
    lambda x: int(
        bool(re.search(r"\b\d{8,15}\b", str(x)))
    )
)

# Possui valor monetário
df["possui_valor_monetario"] = df["texto"].apply(
    lambda x: int(
        bool(re.search(
            r"(R\$|\$|MZN|MT)\s?\d+|\d+[.,]?\d*\s?(MZN|MT)",
            str(x),
            re.I
        ))
    )
)

# Vocabulário de urgência/ação
palavras_urgencia = [
    "urgente",
    "agora",
    "imediatamente",
    "prazo",
    "expire",
    "expira",
    "clique",
    "acesse",
    "confirme",
    "bloqueado",
    "bloqueada"
]

df["qtd_urgencia"] = df["texto"].apply(
    lambda x: sum(
        len(re.findall(rf"\b{re.escape(p)}\b", str(x), re.I))
        for p in palavras_urgencia
    )
)

display(df.head())

,id,fonte,texto,classe,qtd_caracteres,qtd_palavras,qtd_numeros,qtd_simbolos,qtd_maiusculas,possui_link,possui_email,possui_telefone,possui_valor_monetario,qtd_urgencia
0,1_0,sms,"Oi, peço para transferir aquele valor para meu...",Legitimate,71,12,0,2,0,0,0,0,0,0
1,2_0,sms,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate,85,18,2,7,0,0,0,0,0,0
2,3_0,sms,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate,153,26,19,13,4,0,0,0,1,0
3,4_0,sms,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate,189,32,21,11,2,0,0,0,1,1
4,5_0,sms,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate,47,9,0,3,0,0,0,0,0,0


Tem duplicatas?

In [14]:
duplicadas = df[df["texto"].duplicated(keep=False)].copy()

print("Registros envolvidos em duplicatas:", len(duplicadas))

display(
    duplicadas[
        ["id", "fonte", "texto", "classe"]
    ].sort_values("texto")
)

Registros envolvidos em duplicatas: 10


,id,fonte,texto,classe
2159,157_1,sms,Aquele Valor Podes Transferir Para Esta Conta ...,Smishing
2555,616_1,sms,Aquele Valor Podes Transferir Para Esta Conta ...,Smishing
549,558_0,sms,"Boa tarde, podes enviar aquele valor para este...",Legitimate
2426,459_1,sms,"Boa tarde, podes enviar aquele valor para este...",Smishing
1524,1555_0,sms,Bom dia O Valor Manda Para Esse Numero 8467727...,Legitimate
2221,223_1,sms,Bom dia O Valor Manda Para Esse Numero 8467727...,Smishing
2210,212_1,sms,ESSE VALOR MANDAR ESTE NUMERO 857426561 NOME D...,Smishing
2365,383_1,sms,ESSE VALOR MANDAR ESTE NUMERO 857426561 NOME D...,Smishing
1303,1329_0,sms,"Manda o valor neste número,858148365.M-pesa ve...",Legitimate
2461,503_1,sms,"Manda o valor neste número,858148365.M-pesa ve...",Smishing


Estatística descritiva das variáveis quantitativas. média, mediana, desvio padrão e distribuição.

In [3]:
variaveis_numericas = [
    "qtd_caracteres",
    "qtd_palavras",
    "qtd_numeros",
    "qtd_simbolos",
    "qtd_maiusculas",
    "qtd_urgencia"
]

estatisticas = df[variaveis_numericas].describe().T

estatisticas["mediana"] = df[variaveis_numericas].median()

estatisticas = estatisticas[
    ["count", "mean", "mediana", "std", "min", "25%", "50%", "75%", "max"]
]

display(estatisticas.round(2))

,count,mean,mediana,std,min,25%,50%,75%,max
qtd_caracteres,3405.0,270.06,117.0,525.40,5.0,70.0,117.0,176.0,7994.0
qtd_palavras,3405.0,42.85,19.0,81.16,1.0,12.0,19.0,30.0,1213.0
qtd_numeros,3405.0,10.99,7.0,23.25,0.0,0.0,7.0,16.0,944.0
qtd_simbolos,3405.0,13.82,5.0,33.68,0.0,2.0,5.0,13.0,699.0
qtd_maiusculas,3405.0,1.91,0.0,3.32,0.0,0.0,0.0,2.0,27.0
qtd_urgencia,3405.0,0.29,0.0,0.76,0.0,0.0,0.0,0.0,16.0


Estatística das variáveis binárias/categóricas. Para possui_link, por exemplo, não faz tanto sentido apresentar desvio padrão como informação principal, então apresentei como quantidade + porcentagem:


Isso prepara as perguntas 2, 7, 10, 16 e 17!! (Whatsapp do grupo)

In [4]:
variaveis_binarias = [
    "possui_link",
    "possui_email",
    "possui_telefone",
    "possui_valor_monetario"
]

resumo_binarias = []

for coluna in variaveis_binarias:

    quantidade = df[coluna].sum()
    percentual = df[coluna].mean() * 100

    resumo_binarias.append({
        "variavel": coluna,
        "quantidade": quantidade,
        "percentual": percentual
    })

resumo_binarias = pd.DataFrame(resumo_binarias)

display(resumo_binarias.round(2))

,variavel,quantidade,percentual
0,possui_link,134,3.94
1,possui_email,46,1.35
2,possui_telefone,945,27.75
3,possui_valor_monetario,941,27.64


Descrição por classe - entender a distribuição de cada variável

Mas ainda sem entrar em correlação ou afirmar associação.

In [5]:
resumo_por_classe = (
    df.groupby("classe")[variaveis_numericas]
      .agg(["mean", "median", "std", "min", "max"])
      .round(2)
)

display(resumo_por_classe)

qtd_caracteres                          qtd_palavras         \
                     mean median     std min   max         mean median   
classe                                                                   
Legitimate         104.12   94.0   59.60   5   477        18.05   17.0   
Smishing           508.86  154.5  756.14  18  7994        78.54   24.0   

                              ... qtd_maiusculas                       \
               std min   max  ...           mean median   std min max   
classe                        ...                                       
Legitimate    9.93   1    83  ...           1.51    0.0  2.40   0  26   
Smishing    117.34   2  1213  ...           2.49    1.0  4.24   0  27   

           qtd_urgencia                       
                   mean median   std min max  
classe                                        
Legitimate         0.04    0.0  0.20   0   2  
Smishing           0.66    0.0  1.07   0  16  

[2 rows x 30 columns]